In [5]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
import pandas as pd
from pathlib import Path

# ---------------- USER INPUTS ----------------
r_compound = r"D:\Phd Research\Final_Raster\100yr_compound_flood_base_stat.tif"   # compound depth (m)
r_bathtub  = r"D:\Phd Research\Final_Raster\Bathtub_depth_100yr_surge_SLR.tif"    # bathtub depth (m)
r_dem      = r"D:\Phd Research\Final_Raster\land_part_area_DEM.tif"               # DEM (elevation, m)

# DEM elevation bins (meters) to report difference ranges for
DEM_BINS = [(-1.0, 2.0), (2.0, 5.0), (5.0, 9.0)]

# Consider only pixels that are flooded in at least one product?
RESTRICT_TO_FLOODED = True
FLOOD_THRESH = 0.10  # m
# ---------------------------------------------------------

def read_band_as_float(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        nod = src.nodata
        if nod is not None:
            arr = np.where(arr == nod, np.nan, arr)
        prof = src.profile
    return arr, prof

def reproject_to_ref(arr, prof_src, prof_ref, resampling=Resampling.bilinear):
    dst = np.full((prof_ref["height"], prof_ref["width"]), np.nan, dtype=np.float32)
    reproject(
        source=arr,
        destination=dst,
        src_transform=prof_src["transform"], src_crs=prof_src["crs"],
        dst_transform=prof_ref["transform"], dst_crs=prof_ref["crs"],
        resampling=resampling,
        src_nodata=np.nan, dst_nodata=np.nan
    )
    return dst

# 1) Read DEM as reference grid
DEM, prof_dem = read_band_as_float(r_dem)

# 2) Read depths and align to DEM grid
D_comp, prof_comp = read_band_as_float(r_compound)
D_bath, prof_bath = read_band_as_float(r_bathtub)

D_comp_on_dem = reproject_to_ref(D_comp, prof_comp, prof_dem, resampling=Resampling.bilinear)
D_bath_on_dem = reproject_to_ref(D_bath, prof_bath, prof_dem, resampling=Resampling.bilinear)

# 3) Valid mask
valid = np.isfinite(DEM) & np.isfinite(D_comp_on_dem) & np.isfinite(D_bath_on_dem)

# Optionally restrict to places flooded in either product
if RESTRICT_TO_FLOODED:
    flooded = (D_comp_on_dem >= FLOOD_THRESH) | (D_bath_on_dem >= FLOOD_THRESH)
    valid &= flooded

# 4) Difference (compound - bathtub)
delta = np.full_like(DEM, np.nan, dtype=np.float32)
delta[valid] = D_comp_on_dem[valid] - D_bath_on_dem[valid]

# 5) Summarize delta ranges per DEM bin
rows = []
for lo, hi in DEM_BINS:
    m = valid & (DEM >= lo) & (DEM <= hi)
    n = int(np.count_nonzero(m))
    if n == 0:
        rows.append({
            "DEM_bin_m": f"[{lo}, {hi}]",
            "pixel_count": 0,
            "delta_min_m": np.nan,
            "delta_p05_m": np.nan,
            "delta_p50_m": np.nan,
            "delta_p95_m": np.nan,
            "delta_p98_m": np.nan,
            "delta_p99_m": np.nan,
            "delta_max_m": np.nan
        })
        continue

    vals = delta[m]
    rows.append({
        "DEM_bin_m": f"[{lo}, {hi}]",
        "pixel_count": n,
        "delta_min_m": float(np.nanmin(vals)),
        "delta_p05_m": float(np.nanpercentile(vals, 5)),
        "delta_p50_m": float(np.nanmedian(vals)),
        "delta_p95_m": float(np.nanpercentile(vals, 95)),
        "delta_p98_m": float(np.nanpercentile(vals, 98)),
        "delta_p99_m": float(np.nanpercentile(vals, 99)),
        "delta_max_m": float(np.nanmax(vals))
    })

df = pd.DataFrame(rows)
print("\nDifference ranges Δ = (compound − bathtub) by DEM elevation bin:\n")
print(df.to_string(index=False))

# 6) Save CSV (optional)
out_csv = Path.cwd() / "delta_ranges_by_dem_bins.csv"
df.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")



Difference ranges Δ = (compound − bathtub) by DEM elevation bin:

  DEM_bin_m  pixel_count  delta_min_m  delta_p05_m  delta_p50_m  delta_p95_m  delta_p98_m  delta_p99_m  delta_max_m
[-1.0, 2.0]       135049    -5.968216    -2.869156    -0.404982     1.771423     3.171420     4.495774    14.186012
 [2.0, 5.0]       135713    -5.378860    -2.731064    -0.238760     4.579521     5.711197     6.475112    15.811447
 [5.0, 9.0]        75014    -2.421671    -0.734923     0.403279     6.331906     7.689148     8.516905    15.842480

Saved: C:\Users\sahad2\delta_ranges_by_dem_bins.csv
